# Step 1 Imports

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np

# Step 2 Load Basic data

In [4]:
# aapl = yf.download("AAPL", start="2020-01-01", interval="1d")

start = "2026-03-23"
end = "2026-03-29"
vocab_tickers = ["AAPL", "MSFT", "GOOGL"]
successful = {}
failed = []

t2i = {v:i for i,v in enumerate(vocab_tickers)}
i2t = {i:v for i,v in enumerate(vocab_tickers)}

for ticker in vocab_tickers:
    df = yf.download(ticker,start=start,interval="1d",progress=False,auto_adjust=True)
    if len(df) > 1:
        successful[ticker] = df["Close"][ticker]
        print(f"  ✓ {ticker:12s} {len(df)} days")
    else:
        failed.append(ticker)
        print(f"  ✗ {ticker:12s} insufficient history")

# for k in successful:
#     print(k,":",len(successful[k]))

prices = pd.DataFrame(successful)
returns = prices.pct_change().dropna()

threshold = 0.005
print("Returns")
print(returns)

all_pairs = []

for date, row in returns.iterrows():
    print(f"\n{'='*50}")
    print(f"Date: {date.date()}")
    # for label,value in row.items():
    #     print(f"  {label:12s} {value:+.3f}")
    up = [
        t for t in row.index
        if row[t] > threshold
        and t in t2i
        and not np.isnan(row[t])
    ]

    down = [
        t for t in row.index
        if row[t] < -threshold
        and t in t2i
        and not np.isnan(row[t])
    ]
    # print(f"UP Tickers{'='*50}\n",up)
    # print(f"DOWN Tickers{'='*50}\n",down)

    for group in [up, down]:
        for i in range(len(group)):
            for j in range(i+1, len(group)):
                
                # forward pair
                all_pairs.append((t2i[group[i]],t2i[group[j]]))

                # backward pair
                all_pairs.append((t2i[group[j]], t2i[group[i]]))
print(f"{'#'*20}Training Pairs{'#'*20}")
print(all_pairs)
for pair in all_pairs:
    print(pair,"::",i2t[pair[0]], i2t[pair[1]])
    



  ✓ AAPL         15 days
  ✓ MSFT         15 days
  ✓ GOOGL        15 days
Returns
                AAPL      MSFT     GOOGL
Date                                    
2026-03-24  0.000596 -0.026789 -0.038469
2026-03-25  0.003894 -0.004561  0.001687
2026-03-26  0.001069 -0.013664 -0.034407
2026-03-27 -0.016173 -0.025139 -0.023423
2026-03-30 -0.008722  0.006138 -0.003062
2026-03-31  0.029031  0.031229  0.051408
2026-04-01  0.007250 -0.002161  0.034184
2026-04-02  0.001134  0.011073 -0.005447
2026-04-06  0.011488 -0.001553  0.014268
2026-04-07 -0.020706 -0.001582  0.018234
2026-04-08  0.021302  0.005480  0.038827
2026-04-09  0.006141 -0.003366  0.003687
2026-04-10 -0.000038 -0.005897 -0.003925
2026-04-13 -0.004914  0.036401  0.012829

Date: 2026-03-24

Date: 2026-03-25

Date: 2026-03-26

Date: 2026-03-27

Date: 2026-03-30

Date: 2026-03-31

Date: 2026-04-01

Date: 2026-04-02

Date: 2026-04-06

Date: 2026-04-07

Date: 2026-04-08

Date: 2026-04-09

Date: 2026-04-10

Date: 2026-04-13
#########

In [5]:
S = 5000
ls = len(str(S))

print(
    (lambda x: 10**(x-1)) (ls)
)

x = 10

print(x**(ls-1))

x=1
p = 10
for i in range(ls-1):
    x = x * p
      
print(ls,",",x)

1000
1000
4 , 1000


# Step 3 Prep for Training


In [6]:
V = len(vocab_tickers)
EMBED_DIM = 8
EPOCHS = 5000
LR = 1e-6

W1 = np.random.randn(V, EMBED_DIM) * 0.01
W2 = np.random.randn(EMBED_DIM, V) * 0.01
print("Initial W1 — all near zero, no meaning yet:")
for i in range(V):
    print(f"  {i2t[i]:6s} → {W1[i].round(4)}")

def softmax(x):
    e = np.exp(x - x.max())    # subtract max for stability
    return e / e.sum()

Initial W1 — all near zero, no meaning yet:
  AAPL   → [-0.012   0.0074 -0.0117 -0.0013  0.0088  0.0133  0.0049  0.013 ]
  MSFT   → [ 0.0145 -0.0032  0.002   0.0146 -0.004  -0.0079  0.0073  0.0145]
  GOOGL  → [-0.0243  0.0073 -0.0044  0.0006  0.0005  0.0001 -0.004   0.0088]


# Step 4 Training Loop

In [7]:
losses = []
loss_by_pair = {}
print("Training...\n")
print(f"{'Epoch':>6}  {'Loss':>8}  {'Best pair':>20}  {'Worst pair':>20}")
print("-" * 65)

for epoch in range(EPOCHS):
    total_loss = 0
    epoch_losses = {} # loss per pair this epoch

    indices = np.random.permutation(len(all_pairs))
    
    for idx in indices:
        center_idx, target_idx = all_pairs[idx]

        # Forward pass
        h = W1[center_idx]          # hidden layer
        scores = h@W2
        probs = softmax(scores)

        # Loss

        loss = -np.log(probs[target_idx] + 1e-9)
        total_loss += loss
        pair_key = (i2t[center_idx], i2t[target_idx])
        epoch_losses[pair_key] = epoch_losses.get(pair_key, 0) + loss

        # Backward Pass
        dscores = probs.copy()
        dscores[target_idx] = -1
        dW2 = np.outer(h, dscores)
        dh = W2 @ dscores

        # Update
        W2 -= LR * dW2
        W1[center_idx] -= LR * dh
    
    # Epoch Stats
    avg_loss = total_loss / len(all_pairs)
    losses.append(avg_loss)
    loss_by_pair[epoch] = epoch_losses
    ls = len(str(EPOCHS))
    ed = 10 ** (ls-1)
    if epoch % ed == 0 or epoch == EPOCHS - 1:
        sorted_pairs = sorted(epoch_losses.items(), key=lambda x: x[1])
        best_pair = sorted_pairs[0][0]
        worst_pair = sorted_pairs[-1][0]

        best_str  = f"{best_pair[0]}-{best_pair[1]}"
        worst_str = f"{worst_pair[0]}-{worst_pair[1]}"

        print(f"{epoch:>6}  {avg_loss:>8.4f}  "
              f"{best_str:>20}  {worst_str:>20}")
print("\nTraining Complete\n")

Training...

 Epoch      Loss             Best pair            Worst pair
-----------------------------------------------------------------
     0    1.0986             MSFT-AAPL            MSFT-GOOGL
  1000    1.0986             MSFT-AAPL            MSFT-GOOGL
  2000    1.0986             MSFT-AAPL            MSFT-GOOGL
  3000    1.0986             MSFT-AAPL            MSFT-GOOGL
  4000    1.0986             MSFT-AAPL            MSFT-GOOGL
  4999    1.0986             MSFT-AAPL            MSFT-GOOGL

Training Complete



In [46]:
# ── 5. Extract embeddings ─────────────────────────────────
embeddings = W1.copy()    # W2 discarded — same as Word2Vec

print("Learned embeddings:")
for i in range(V):
    print(f"  {i2t[i]:6s} → {embeddings[i].round(4)}")

Learned embeddings:
  AAPL   → [ 0.0027  0.0219 -0.0067  0.01   -0.0067  0.0188  0.0298 -0.0134]
  MSFT   → [-0.0093  0.0057 -0.0237  0.0056  0.0069 -0.0043 -0.0129  0.0098]
  GOOGL  → [-0.0134  0.0017  0.0162 -0.0035 -0.0051 -0.0021  0.0231  0.0131]
  AMZN   → [ 0.0007 -0.0012 -0.0114  0.0009  0.0121 -0.0111 -0.0009 -0.0021]
  META   → [-0.013   0.0106  0.0001 -0.008   0.0094 -0.0111  0.0103  0.0083]
  TSLA   → [ 0.0009  0.0068  0.0091  0.0124  0.0223 -0.0014 -0.0215 -0.0114]
  NVDA   → [-0.0081 -0.0139 -0.0115  0.0089  0.0126 -0.0117  0.0154 -0.007 ]


In [47]:
def cosine_sim(a, b):
    dot   = np.dot(a, b)
    norms = np.linalg.norm(a) * np.linalg.norm(b)
    return dot / norms if norms > 0 else 0.0

In [48]:
def most_similar(ticker, top_n=None):
    """
    Find stocks most similar to ticker
    by cosine similarity of their embeddings.
    """
    if ticker not in t2i:
        print(f"{ticker} not in vocab")
        return []
    
    vec    = embeddings[t2i[ticker]]
    scores = [
        (i2t[i], cosine_sim(vec, embeddings[i]))
        for i in range(V)
        if i2t[i] != ticker
    ]
    scores = sorted(scores, key=lambda x: -x[1])
    return scores[:top_n] if top_n else scores

print("Most similar stocks:\n")
for ticker in vocab_tickers:
    similar = most_similar(ticker, top_n=3)
    result  = "  ".join([f"{t}({s:.3f})" for t, s in similar])
    print(f"  {ticker:6s} → {result}")

Most similar stocks:

  AAPL   → GOOGL(0.233)  NVDA(0.060)  META(0.033)
  MSFT   → AMZN(0.587)  META(0.222)  NVDA(0.178)
  GOOGL  → META(0.583)  AAPL(0.233)  NVDA(0.081)
  AMZN   → NVDA(0.662)  MSFT(0.587)  META(0.327)
  META   → GOOGL(0.583)  AMZN(0.327)  NVDA(0.266)
  TSLA   → AMZN(0.306)  MSFT(0.176)  NVDA(-0.042)
  NVDA   → AMZN(0.662)  META(0.266)  MSFT(0.178)
